# Encrypted Machine Learning: Preprocessing (`ml/preprocessing.py`)

This tutorial covers `src/concrete_fhe_toolkit/ml/preprocessing.py`. Preprocessing transformers can be chained inside an `FHEPipeline` before a model. They operate entirely in the encrypted domain, allowing for secure standard scaling, min-max scaling, and continuous-to-categorical binning.

## 1. Encrypted Binning (Scorecard binning)

An `FHEBinner` takes continuous encrypted inputs and assigns them to ordinal bucket indices based on public bucket boundaries. This is especially useful for logistic regression scorecards.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.preprocessing import FHEBinner

def test_binner(age: int, salary: int):
    binner = FHEBinner([
        [18, 30, 45, 65],     # Age boundaries
        [1000, 5000, 20000]   # Salary boundaries
    ])
    return binner._transform_logic([age, salary])

compiler = fhe.Compiler(test_binner, {"age": "encrypted", "salary": "encrypted"})
circuit = compiler.compile([(1, 1), (100, 100000)])

# Age 35 is in Bin 2: [30, 45)
# Salary 7000 is in Bin 2: [5000, 20000)
assert circuit.encrypt_run_decrypt(35, 7000) == [2, 2]

# Age 17 is in Bin 0: (-inf, 18)
# Salary 900 is in Bin 0: (-inf, 1000)
assert circuit.encrypt_run_decrypt(17, 900) == [0, 0]

print("✅ Encrypted FHEBinner passed!")

## 2. Standard Scaler

Computes z-scores `((x - mean) * scale) // std` per feature while remaining in the integer domain.

In [ ]:
from concrete_fhe_toolkit.ml.preprocessing import FHEStandardScaler

def test_scaler(feat1: int, feat2: int):
    # Feature 1: mean 50, std 10
    # Feature 2: mean 100, std 20
    scaler = FHEStandardScaler(means=[50, 100], stds=[10, 20], scale=10)
    return scaler._transform_logic([feat1, feat2])

compiler = fhe.Compiler(test_scaler, {"feat1": "encrypted", "feat2": "encrypted"})
circuit = compiler.compile([(50, 100)])

# Feature 1: (65 - 50) / 10 = 1.5 -> Scaled by 10 -> 15
# Feature 2: (60 - 100) / 20 = -2.0 -> Scaled by 10 -> -20
assert circuit.encrypt_run_decrypt(65, 60) == [15, -20]
print("✅ Encrypted FHEStandardScaler passed!")